# Vector Store Base Interfaces Reference

Developer-facing statements defined in `langchain_core.vectorstores.base`.


# `VectorStore: ABC`

Abstract interface for storing embedded data and performing vector search.

A concrete subclass must implement `similarity_search` and `from_texts`. To support mutation, retrieval by ID, scored search, vector search, maximal marginal relevance, or native asynchronous operations, it should override the corresponding optional hooks.

## Properties

### `embeddings`

Returns the query embedding object when the store exposes one. The base implementation returns `None`.

```python
embeddings: Embeddings | None
```

## Required subclass hooks

### `similarity_search`

```python
similarity_search(
    self,
    query: str, # Input text
    k: int = 4, # Maximum number of documents to return
    **kwargs: Any, # Store-specific search arguments
) -> list[Document] # Documents most similar to the query
```

### `from_texts`

```python
@classmethod
from_texts(
    cls: type[VST],
    texts: list[str], # Texts used to initialize the store
    embedding: Embeddings, # Embedding implementation to use
    metadatas: list[dict[str, Any]] | None = None, # Optional metadata for each text
    *,
    ids: list[str] | None = None, # Optional IDs for the texts
    **kwargs: Any, # Store-specific initialization arguments
) -> VST # Initialized vector store
```


## Adding data

### `add_texts`

Embeds and adds text values. When a subclass overrides `add_documents`, the base implementation converts the texts into `Document` objects and delegates to it; otherwise, it raises `NotImplementedError`.

```python
add_texts(
    self,
    texts: Iterable[str], # Text values to add
    metadatas: list[dict[str, Any]] | None = None, # Optional metadata for each text
    *,
    ids: list[str] | None = None, # Optional text IDs
    **kwargs: Any, # Store-specific arguments
) -> list[str] # IDs assigned to the added texts
```

Raises `ValueError` when the supplied metadata count does not match the text count. The source documentation also declares a `ValueError` for an ID-count mismatch.

### `aadd_texts`

Asynchronous counterpart to `add_texts`. It delegates to an overridden `aadd_documents` when available; otherwise, it runs `add_texts` through an executor.

```python
async aadd_texts(
    self,
    texts: Iterable[str],
    metadatas: list[dict[str, Any]] | None = None,
    *,
    ids: list[str] | None = None,
    **kwargs: Any,
) -> list[str]
```

### `add_documents`

Adds or updates documents. When a subclass overrides `add_texts`, the base implementation extracts page content, metadata, and usable document IDs and delegates to it; otherwise, it raises `NotImplementedError`. IDs passed through `kwargs` take precedence over IDs stored on documents.

```python
add_documents(
    self,
    documents: list[Document], # Documents to add or update
    **kwargs: Any, # Store-specific arguments
) -> list[str] # IDs assigned to the documents
```

### `aadd_documents`

Asynchronous counterpart to `add_documents`. It delegates to an overridden `aadd_texts` when available; otherwise, it runs `add_documents` through an executor.

```python
async aadd_documents(
    self,
    documents: list[Document],
    **kwargs: Any,
) -> list[str]
```

In [ ]:
#%pip install -U langchain-core # Install LangChain Core if it is not installed

from collections.abc import Iterable # Import Iterable for accepting multiple text values
from typing import Any # Import Any for additional vector-store arguments
from uuid import uuid4 # Import uuid4 for generating unique IDs
from langchain_core.documents import Document # Import Document for storing text and metadata
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.embeddings.fake import DeterministicFakeEmbedding # Import a test embedding model
from langchain_core.vectorstores import VectorStore # Import the abstract vector-store class


class SimpleVectorStore(VectorStore): # Create a basic in-memory vector store

    def __init__(self, embedding: Embeddings) -> None: # Initialize the vector store
        self._embedding = embedding # Store the embedding model
        self.documents: dict[str, Document] = {} # Store documents using their IDs
        self.vectors: dict[str, list[float]] = {} # Store embedding vectors using document IDs

    @property # Define the embeddings property
    def embeddings(self) -> Embeddings: # Return the configured embedding model
        return self._embedding # Return the stored embedding model

    def add_texts( # Define synchronous text addition
        self, # Receive the current vector-store object
        texts: Iterable[str], # Receive text values to add
        metadatas: list[dict[str, Any]] | None = None, # Receive optional metadata
        *, # Make the following arguments keyword-only
        ids: list[str] | None = None, # Receive optional document IDs
        **kwargs: Any, # Receive additional store-specific arguments
    ) -> list[str]: # Return the assigned document IDs
        text_list = list(texts) # Convert the text iterable into a list

        if metadatas is None: # Check whether metadata was omitted
            metadatas = [{} for _ in text_list] # Create empty metadata for every text

        if len(metadatas) != len(text_list): # Check whether metadata and text counts match
            raise ValueError("Metadata count must match text count.") # Raise an error for mismatched counts

        if ids is None: # Check whether IDs were omitted
            ids = [str(uuid4()) for _ in text_list] # Generate one unique ID for every text

        if len(ids) != len(text_list): # Check whether ID and text counts match
            raise ValueError("ID count must match text count.") # Raise an error for mismatched counts

        generated_vectors = self._embedding.embed_documents(text_list) # Embed all supplied texts

        for text, metadata, document_id, vector in zip( # Process each text and its related values
            text_list, # Supply the text values
            metadatas, # Supply the metadata values
            ids, # Supply the document IDs
            generated_vectors, # Supply the generated vectors
        ): # Finish combining the values
            document = Document( # Create a LangChain Document
                id=document_id, # Assign the document ID
                page_content=text, # Store the supplied text
                metadata=metadata, # Store the supplied metadata
            ) # Finish creating the Document

            self.documents[document_id] = document # Store the Document by ID
            self.vectors[document_id] = vector # Store the vector by ID

        return ids # Return all assigned IDs

    def similarity_search( # Implement the required search method
        self, # Receive the current vector-store object
        query: str, # Receive the search query
        k: int = 4, # Receive the maximum result count
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return matching documents
        return list(self.documents.values())[:k] # Return the first available documents for this simple demo

    @classmethod # Define a class-level factory method
    def from_texts( # Create a vector store from text values
        cls, # Receive the vector-store class
        texts: list[str], # Receive initial text values
        embedding: Embeddings, # Receive the embedding model
        metadatas: list[dict[str, Any]] | None = None, # Receive optional metadata
        *, # Make the following arguments keyword-only
        ids: list[str] | None = None, # Receive optional IDs
        **kwargs: Any, # Receive additional initialization arguments
    ) -> "SimpleVectorStore": # Return an initialized vector store
        store = cls(embedding=embedding) # Create an empty vector store
        store.add_texts(texts=texts, metadatas=metadatas, ids=ids) # Add the supplied texts
        return store # Return the populated vector store


embedding_model = DeterministicFakeEmbedding(size=4) # Create a deterministic four-dimensional embedding model
store = SimpleVectorStore(embedding=embedding_model) # Create the custom vector store

In [ ]:
# Synchronous add_texts()
text_ids = store.add_texts( # Add plain text values synchronously
    texts=[ # Provide the text values
        "Python is a programming language.", # Add the first text
        "LangChain helps build LLM applications.", # Add the second text
    ], # Finish the text list
    metadatas=[ # Provide metadata for each text
        {"source": "python.txt"}, # Add metadata for the first text
        {"source": "langchain.txt"}, # Add metadata for the second text
    ], # Finish the metadata list
    ids=["text-1", "text-2"], # Provide explicit IDs
) # Finish adding the texts

print(f"Text IDs: {text_ids}") # Display the assigned text IDs

In [ ]:
# 2. Synchronous add_documents()
documents = [ # Create Documents to add
    Document( # Create the first Document
        id="doc-1", # Assign the first document ID
        page_content="Vector stores save document embeddings.", # Store the first document text
        metadata={"source": "vectorstore.txt"}, # Store the first document metadata
    ), # Finish the first Document
    Document( # Create the second Document
        id="doc-2", # Assign the second document ID
        page_content="Retrievers find relevant documents.", # Store the second document text
        metadata={"source": "retriever.txt"}, # Store the second document metadata
    ), # Finish the second Document
] # Finish the Document list

document_ids = store.add_documents(documents=documents) # Add Documents through the inherited method
print(f"Document IDs: {document_ids}") # Display the assigned document IDs

In [ ]:
# 3. Asynchronous aadd_texts()
async_text_ids = await store.aadd_texts( # Add text values asynchronously
    texts=[ # Provide asynchronous text values
        "Embeddings convert text into vectors.", # Add the first asynchronous text
        "Similarity search compares query vectors.", # Add the second asynchronous text
    ], # Finish the text list
    metadatas=[ # Provide metadata for the asynchronous texts
        {"source": "embedding.txt"}, # Add metadata for the first text
        {"source": "search.txt"}, # Add metadata for the second text
    ], # Finish the metadata list
    ids=["async-text-1", "async-text-2"], # Provide explicit asynchronous text IDs
) # Finish asynchronously adding texts

print(f"Async text IDs: {async_text_ids}") # Display the asynchronously assigned IDs

In [ ]:
# 4. Asynchronous aadd_documents()
async_documents = [ # Create Documents for asynchronous addition
    Document( # Create the first asynchronous Document
        id="async-doc-1", # Assign the first asynchronous document ID
        page_content="Async methods prevent blocking operations.", # Store the first document text
        metadata={"source": "async.txt"}, # Store the first document metadata
    ), # Finish the first asynchronous Document
    Document( # Create the second asynchronous Document
        id="async-doc-2", # Assign the second asynchronous document ID
        page_content="Executors run synchronous methods asynchronously.", # Store the second document text
        metadata={"source": "executor.txt"}, # Store the second document metadata
    ), # Finish the second asynchronous Document
] # Finish the asynchronous Document list

async_document_ids = await store.aadd_documents(documents=async_documents) # Add Documents asynchronously
print(f"Async document IDs: {async_document_ids}") # Display the asynchronously assigned document IDs

In [ ]:
# Display all stored data
print(f"\nTotal stored documents: {len(store.documents)}") # Display the total number of stored Documents
print("=" * 70) # Display a separator

for document_id, document in store.documents.items(): # Process every stored Document
    print(f"ID: {document_id}") # Display the document ID
    print(f"Content: {document.page_content}") # Display the document content
    print(f"Metadata: {document.metadata}") # Display the document metadata
    print(f"Vector: {store.vectors[document_id]}") # Display the generated vector
    print("-" * 70) # Display a separator after each Document

## Deletion and ID lookup

### `delete`

```python
delete(
    self,
    ids: list[str] | None = None, # IDs to delete; `None` may represent all entries
    **kwargs: Any, # Additional deletion criteria
) -> bool | None # Success status, or `None` when unsupported
```

The base implementation raises `NotImplementedError`.

### `adelete`

Runs `delete` through an executor.

```python
async adelete(
    self,
    ids: list[str] | None = None,
    **kwargs: Any,
) -> bool | None
```

### `get_by_ids`

Returns documents whose `id` fields identify the matching vector-store entries. Missing or duplicate IDs may result in fewer documents, and result order is not required to match input order. Implementations should not fail merely because some IDs are missing.

```python
get_by_ids(
    self,
    ids: Sequence[str], # IDs to retrieve
    /,
) -> list[Document] # Matching documents
```

The base implementation raises `NotImplementedError`.

### `aget_by_ids`

Runs `get_by_ids` through an executor. Subclasses may override it with native asynchronous behaviour.

```python
async aget_by_ids(
    self,
    ids: Sequence[str],
    /,
) -> list[Document]
```

In [ ]:
# create a vectorstore

#%pip install -U langchain-core # Install LangChain Core if it is unavailable

from collections.abc import Sequence # Import Sequence for the ID lookup parameter
from typing import Any # Import Any for additional method arguments
from uuid import uuid4 # Import uuid4 for generating document IDs
from langchain_core.documents import Document # Import the LangChain Document class
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.embeddings.fake import DeterministicFakeEmbedding # Import a test embedding model
from langchain_core.vectorstores import VectorStore # Import the abstract vector-store class


class InMemoryVectorStore(VectorStore): # Create a simple in-memory vector store

    def __init__(self, embedding: Embeddings) -> None: # Initialize the vector store
        self._embedding = embedding # Store the embedding model
        self.documents: dict[str, Document] = {} # Store documents using their IDs
        self.vectors: dict[str, list[float]] = {} # Store vectors using document IDs

    @property # Define the embedding-model property
    def embeddings(self) -> Embeddings: # Return the configured embedding model
        return self._embedding # Return the stored embedding model

    def add_documents(self, documents: list[Document], **kwargs: Any) -> list[str]: # Add documents to the store
        assigned_ids = [] # Create a list for assigned document IDs

        for document in documents: # Process every supplied document
            document_id = document.id or str(uuid4()) # Use the existing ID or generate a new ID
            stored_document = Document( # Create a document with a guaranteed ID
                id=document_id, # Assign the resolved document ID
                page_content=document.page_content, # Copy the document content
                metadata=document.metadata, # Copy the document metadata
            ) # Finish creating the stored document
            vector = self._embedding.embed_query(document.page_content) # Generate the document vector
            self.documents[document_id] = stored_document # Store the document by ID
            self.vectors[document_id] = vector # Store the corresponding vector
            assigned_ids.append(document_id) # Record the assigned ID

        return assigned_ids # Return all assigned IDs

    def get_by_ids(self, ids: Sequence[str], /) -> list[Document]: # Retrieve documents using their IDs
        requested_ids = set(ids) # Remove duplicate requested IDs
        matching_documents = [] # Create a list for matching documents

        for document_id, document in self.documents.items(): # Process every stored document
            if document_id in requested_ids: # Check whether its ID was requested
                matching_documents.append(document) # Add the matching document

        return matching_documents # Return the documents that were found

    def delete(self, ids: list[str] | None = None, **kwargs: Any) -> bool: # Delete selected or all documents
        if ids is None: # Check whether all documents should be deleted
            self.documents.clear() # Remove every stored document
            self.vectors.clear() # Remove every stored vector
            return True # Report successful deletion

        for document_id in ids: # Process every supplied ID
            self.documents.pop(document_id, None) # Remove the document without failing for missing IDs
            self.vectors.pop(document_id, None) # Remove the vector without failing for missing IDs

        return True # Report successful deletion

    def similarity_search(self, query: str, k: int = 4, **kwargs: Any) -> list[Document]: # Implement the required search method
        return list(self.documents.values())[:k] # Return the first available documents for this focused example

    @classmethod # Define a class-level factory method
    def from_texts( # Create a vector store from text values
        cls, # Receive the vector-store class
        texts: list[str], # Receive the initial text values
        embedding: Embeddings, # Receive the embedding model
        metadatas: list[dict[str, Any]] | None = None, # Receive optional metadata
        *, # Make the remaining argument keyword-only
        ids: list[str] | None = None, # Receive optional document IDs
        **kwargs: Any, # Receive additional initialization arguments
    ) -> "InMemoryVectorStore": # Return an initialized vector store
        metadatas = metadatas or [{} for _ in texts] # Create empty metadata when none is supplied
        ids = ids or [str(uuid4()) for _ in texts] # Generate IDs when none are supplied
        documents = [ # Create Documents from the supplied values
            Document(id=document_id, page_content=text, metadata=metadata) # Create one Document
            for text, metadata, document_id in zip(texts, metadatas, ids) # Combine corresponding values
        ] # Finish creating the Document list
        store = cls(embedding=embedding) # Create an empty vector store
        store.add_documents(documents) # Add the created Documents
        return store # Return the populated vector store

In [ ]:
# Populate the store

embedding_model = DeterministicFakeEmbedding(size=5) # Create a deterministic test embedding model

store = InMemoryVectorStore.from_texts( # Create and populate the vector store
    texts=[ # Provide the document text values
        "Python is a programming language.", # Provide the first document
        "LangChain helps build LLM applications.", # Provide the second document
        "Vector stores save document embeddings.", # Provide the third document
    ], # Finish the text list
    embedding=embedding_model, # Supply the embedding model
    metadatas=[ # Provide metadata for each document
        {"source": "python.txt"}, # Provide metadata for the first document
        {"source": "langchain.txt"}, # Provide metadata for the second document
        {"source": "vectorstore.txt"}, # Provide metadata for the third document
    ], # Finish the metadata list
    ids=["doc-1", "doc-2", "doc-3"], # Provide explicit document IDs
) # Finish creating the vector store

print(f"Stored IDs: {list(store.documents)}") # Display all stored document IDs

In [ ]:
# Synchronous ID lookup
found_documents = store.get_by_ids( # Retrieve selected documents
    ["doc-3", "missing-id", "doc-1", "doc-1"] # Include valid, missing, and duplicate IDs
) # Finish retrieving documents

print(f"Documents found: {len(found_documents)}") # Display the number of unique matching documents

for document in found_documents: # Process every matching document
    print(f"ID: {document.id}") # Display the document ID
    print(f"Content: {document.page_content}") # Display the document content
    print("-" * 50) # Display a separator

In [ ]:
# Synchronous deletion
delete_result = store.delete(ids=["doc-2", "missing-id"]) # Delete one existing and one missing ID
print(f"Deletion successful: {delete_result}") # Display the deletion status
print(f"Remaining IDs: {list(store.documents)}") # Display the remaining document IDs

In [ ]:
# Asynchronous ID lookup
async_documents = await store.aget_by_ids(["doc-1", "doc-3"]) # Retrieve documents asynchronously

for document in async_documents: # Process every asynchronously retrieved document
    print(f"{document.id}: {document.page_content}") # Display the document ID and content

In [ ]:
# Asynchronous deletion
async_delete_result = await store.adelete(ids=["doc-3"]) # Delete a document asynchronously
print(f"Async deletion successful: {async_delete_result}") # Display the deletion result
print(f"Remaining IDs: {list(store.documents)}") # Display the remaining document IDs

In [ ]:
# Delete everything
store.delete(ids=None) # Delete all documents and vectors
print(f"Remaining documents: {store.documents}") # Confirm that no documents remain
print(f"Remaining vectors: {store.vectors}") # Confirm that no vectors remain

## Search dispatch

### `search`

Dispatches to similarity search, relevance-threshold search, or maximal marginal relevance search.

```python
search(
    self,
    query: str, # Input text
    search_type: str, # `"similarity"`, `"similarity_score_threshold"`, or `"mmr"`
    **kwargs: Any, # Arguments forwarded to the selected search method
) -> list[Document] # Retrieved documents
```

Raises `ValueError` for an unsupported search type.

### `asearch`

Asynchronous search dispatcher with the same supported search types.

```python
async asearch(
    self,
    query: str,
    search_type: str,
    **kwargs: Any,
) -> list[Document]
```

In [ ]:
#%pip install -U langchain-core # Install LangChain Core if it is unavailable

import re # Import regular expressions for extracting words
from typing import Any # Import Any for additional search arguments
from langchain_core.documents import Document # Import the LangChain Document class
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.embeddings.fake import DeterministicFakeEmbedding # Import a test embedding model
from langchain_core.vectorstores import VectorStore # Import the abstract VectorStore class


def extract_words(text: str) -> set[str]: # Convert text into a set of lowercase words
    return set(re.findall(r"\b\w+\b", text.lower())) # Extract and return unique words


def text_similarity(first_text: str, second_text: str) -> float: # Calculate Jaccard similarity between two texts
    first_words = extract_words(first_text) # Extract words from the first text
    second_words = extract_words(second_text) # Extract words from the second text
    combined_words = first_words.union(second_words) # Find all unique words from both texts

    if not combined_words: # Check whether both texts are empty
        return 0.0 # Return zero similarity for empty texts

    matching_words = first_words.intersection(second_words) # Find words present in both texts
    return len(matching_words) / len(combined_words) # Return the normalized similarity score


class FAQVectorStore(VectorStore): # Create an in-memory vector store for FAQ documents

    def __init__(self, documents: list[Document], embedding: Embeddings) -> None: # Initialize the vector store
        self.documents = documents # Store the FAQ documents
        self._embedding = embedding # Store the embedding model

    @property # Define the embeddings property
    def embeddings(self) -> Embeddings: # Return the configured embedding model
        return self._embedding # Return the stored embedding model

    def similarity_search_with_relevance_scores( # Define relevance-scored search
        self, # Receive the current vector-store object
        query: str, # Receive the user query
        k: int = 4, # Receive the maximum number of results
        **kwargs: Any, # Receive optional search arguments
    ) -> list[tuple[Document, float]]: # Return documents with normalized scores
        score_threshold = kwargs.get("score_threshold") # Read an optional minimum relevance score
        scored_documents = [] # Create a list for documents and their scores

        for document in self.documents: # Process every stored document
            score = text_similarity(query, document.page_content) # Calculate query-document similarity
            scored_documents.append((document, score)) # Store the document with its score

        scored_documents.sort(key=lambda item: item[1], reverse=True) # Sort results by descending score
        scored_documents = scored_documents[:k] # Keep only the top-k results

        if score_threshold is not None: # Check whether threshold filtering was requested
            scored_documents = [ # Create a filtered result list
                (document, score) # Preserve each qualifying document and score
                for document, score in scored_documents # Process every scored document
                if score >= score_threshold # Keep only scores meeting the threshold
            ] # Finish the filtered result list

        return scored_documents # Return the scored search results

    def similarity_search( # Define normal similarity search
        self, # Receive the current vector-store object
        query: str, # Receive the user query
        k: int = 4, # Receive the maximum number of results
        **kwargs: Any, # Receive optional search arguments
    ) -> list[Document]: # Return the most similar documents
        scored_documents = self.similarity_search_with_relevance_scores( # Perform scored similarity search
            query=query, # Forward the query
            k=k, # Forward the maximum result count
        ) # Finish the scored search
        return [document for document, score in scored_documents] # Remove scores and return only documents

    def max_marginal_relevance_search( # Define maximal marginal relevance search
        self, # Receive the current vector-store object
        query: str, # Receive the user query
        k: int = 4, # Receive the number of documents to select
        fetch_k: int = 20, # Receive the number of initial candidates
        lambda_mult: float = 0.5, # Control the balance between relevance and diversity
        **kwargs: Any, # Receive optional search arguments
    ) -> list[Document]: # Return relevant and diverse documents
        candidates = self.similarity_search_with_relevance_scores( # Retrieve initial candidates
            query=query, # Forward the user query
            k=fetch_k, # Retrieve up to fetch_k candidates
        ) # Finish retrieving candidates

        if not candidates: # Check whether no candidate documents were found
            return [] # Return an empty result list

        selected_documents = [candidates[0][0]] # Select the most relevant document first
        remaining_documents = [document for document, score in candidates[1:]] # Store the remaining candidates

        while remaining_documents and len(selected_documents) < k: # Continue until enough documents are selected
            best_document = None # Create a placeholder for the next selected document
            best_mmr_score = float("-inf") # Start with the lowest possible MMR score

            for candidate in remaining_documents: # Evaluate every remaining candidate
                query_score = text_similarity(query, candidate.page_content) # Measure query relevance
                diversity_penalty = max( # Find the highest similarity to selected documents
                    text_similarity(candidate.page_content, selected.page_content) # Compare candidate with one selected document
                    for selected in selected_documents # Process every selected document
                ) # Finish calculating the diversity penalty
                mmr_score = (lambda_mult * query_score) - ((1 - lambda_mult) * diversity_penalty) # Calculate the MMR score

                if mmr_score > best_mmr_score: # Check whether this candidate has the best score so far
                    best_mmr_score = mmr_score # Store the new best MMR score
                    best_document = candidate # Store the new best candidate

            selected_documents.append(best_document) # Add the selected diverse document
            remaining_documents.remove(best_document) # Remove it from the candidate list

        return selected_documents # Return the final MMR-selected documents

    @classmethod # Define a class-level factory method
    def from_texts( # Create the vector store from text values
        cls, # Receive the current class
        texts: list[str], # Receive the text values
        embedding: Embeddings, # Receive the embedding model
        metadatas: list[dict[str, Any]] | None = None, # Receive optional metadata
        *, # Make the following arguments keyword-only
        ids: list[str] | None = None, # Receive optional document IDs
        **kwargs: Any, # Receive additional initialization arguments
    ) -> "FAQVectorStore": # Return the initialized vector store
        metadatas = metadatas or [{} for _ in texts] # Create empty metadata when none is supplied
        ids = ids or [f"doc-{index}" for index in range(1, len(texts) + 1)] # Generate IDs when none are supplied
        documents = [ # Create Document objects from the supplied values
            Document(id=document_id, page_content=text, metadata=metadata) # Create one Document
            for text, metadata, document_id in zip(texts, metadatas, ids) # Combine matching values
        ] # Finish creating the document list
        return cls(documents=documents, embedding=embedding) # Return the populated vector store

In [ ]:
# Create the vector store
embedding_model = DeterministicFakeEmbedding(size=5) # Create a deterministic test embedding model

vector_store = FAQVectorStore.from_texts( # Create and populate the vector store
    texts=[ # Provide the FAQ text values
        "Reset your account password from the account settings page.", # Add the password-reset FAQ
        "Contact support when your account login is not working.", # Add the account-support FAQ
        "Refunds are returned to the original payment method.", # Add the refund FAQ
        "Track your delivery through the order tracking page.", # Add the delivery FAQ
        "Change your account email from the profile settings page.", # Add the account-email FAQ
    ], # Finish the FAQ text list
    embedding=embedding_model, # Supply the embedding model
    metadatas=[ # Provide metadata for each FAQ
        {"category": "password"}, # Add metadata for the password FAQ
        {"category": "support"}, # Add metadata for the support FAQ
        {"category": "refund"}, # Add metadata for the refund FAQ
        {"category": "delivery"}, # Add metadata for the delivery FAQ
        {"category": "profile"}, # Add metadata for the profile FAQ
    ], # Finish the metadata list
    ids=["faq-1", "faq-2", "faq-3", "faq-4", "faq-5"], # Provide document IDs
) # Finish creating the vector store

query = "How can I reset my account password?" # Define the user query

In [ ]:
# 1. Similarity search dispatch
similarity_results = vector_store.search( # Use the synchronous search dispatcher
    query=query, # Provide the user query
    search_type="similarity", # Select standard similarity search
    k=3, # Request the three closest documents
) # Finish the similarity search

print("Similarity search results:") # Display the result heading

for document in similarity_results: # Process every retrieved document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# 2. Relevance-score threshold dispatch
threshold_results = vector_store.search( # Use the synchronous search dispatcher
    query=query, # Provide the user query
    search_type="similarity_score_threshold", # Select threshold-based relevance search
    k=5, # Consider up to five documents
    score_threshold=0.15, # Keep only documents scoring at least 0.15
) # Finish the threshold search

print("\nThreshold search results:") # Display the result heading

for document in threshold_results: # Process every qualifying document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# 3. MMR search dispatch
mmr_results = vector_store.search( # Use the synchronous search dispatcher
    query=query, # Provide the user query
    search_type="mmr", # Select maximal marginal relevance search
    k=3, # Select three final documents
    fetch_k=5, # Consider five initial candidates
    lambda_mult=0.6, # Give slightly more importance to relevance than diversity
) # Finish the MMR search

print("\nMMR search results:") # Display the result heading

for document in mmr_results: # Process every MMR-selected document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# 4. Asynchronous search dispatch
async_results = await vector_store.asearch( # Use the asynchronous search dispatcher
    query="My account login has failed.", # Provide an asynchronous search query
    search_type="similarity", # Select standard similarity search
    k=2, # Request the two closest documents
) # Finish the asynchronous search

print("\nAsynchronous search results:") # Display the result heading

for document in async_results: # Process every asynchronously retrieved document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# 5. Unsupported search type
try: # Start handling the expected error
    vector_store.search( # Call the search dispatcher
        query=query, # Provide the user query
        search_type="keyword", # Provide an unsupported search type
    ) # Finish the invalid search call
except ValueError as error: # Catch the unsupported-search-type error
    print(f"\nError: {error}") # Display the error message

## Scored similarity search

### `similarity_search_with_score`

```python
similarity_search_with_score(
    self,
    *args: Any, # Store-specific positional search arguments
    **kwargs: Any, # Store-specific keyword search arguments
) -> list[tuple[Document, float]] # Documents paired with store-native distance or similarity values
```

The base implementation raises `NotImplementedError`.

### `asimilarity_search_with_score`

Runs `similarity_search_with_score` through an executor.

```python
async asimilarity_search_with_score(
    self,
    *args: Any,
    **kwargs: Any,
) -> list[tuple[Document, float]]
```

### `similarity_search_with_relevance_scores`

Returns documents with normalized relevance scores in the range `[0, 1]`. A `score_threshold` value in `kwargs` filters out lower-scoring results. Scores outside the expected range produce a warning; an empty thresholded result is logged.

```python
similarity_search_with_relevance_scores(
    self,
    query: str, # Input text
    k: int = 4, # Maximum number of scored documents to return
    **kwargs: Any, # Search arguments, optionally including `score_threshold`
) -> list[tuple[Document, float]] # Documents paired with normalized relevance scores
```

The default conversion requires the subclass to provide scored search and a relevance-score selection strategy.

### `asimilarity_search_with_relevance_scores`

Asynchronous counterpart to relevance-scored search.

```python
async asimilarity_search_with_relevance_scores(
    self,
    query: str,
    k: int = 4,
    **kwargs: Any,
) -> list[tuple[Document, float]]
```

### `asimilarity_search`

Runs `similarity_search` through an executor. Subclasses may override it with native asynchronous behaviour.

```python
async asimilarity_search(
    self,
    query: str,
    k: int = 4,
    **kwargs: Any,
) -> list[Document]
```

In [ ]:
#%pip install -U langchain-core # Install LangChain Core if it is unavailable

import math # Import math functions for vector calculations
import re # Import regular expressions for extracting words
from typing import Any, Callable # Import types used by the vector store
from langchain_core.documents import Document # Import the LangChain Document class
from langchain_core.embeddings import Embeddings # Import the abstract Embeddings interface
from langchain_core.vectorstores import VectorStore # Import the abstract VectorStore class


class KeywordEmbeddings(Embeddings): # Create a simple keyword-based embedding model

    def __init__(self, vocabulary: list[str]) -> None: # Initialize the embedding model
        self.vocabulary = [word.lower() for word in vocabulary] # Store vocabulary words in lowercase

    def _create_vector(self, text: str) -> list[float]: # Convert one text value into a numeric vector
        words = re.findall(r"\b\w+\b", text.lower()) # Extract lowercase words from the text
        vector = [float(words.count(word)) for word in self.vocabulary] # Count each vocabulary word
        return vector # Return the generated vector

    def embed_documents(self, texts: list[str]) -> list[list[float]]: # Embed multiple documents
        return [self._create_vector(text) for text in texts] # Generate one vector for each document

    def embed_query(self, text: str) -> list[float]: # Embed one search query
        return self._create_vector(text) # Generate and return the query vector


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float: # Calculate cosine similarity
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b)) # Calculate the vector dot product
    magnitude_a = math.sqrt(sum(value ** 2 for value in vector_a)) # Calculate the first vector magnitude
    magnitude_b = math.sqrt(sum(value ** 2 for value in vector_b)) # Calculate the second vector magnitude

    if magnitude_a == 0 or magnitude_b == 0: # Check whether either vector contains only zeros
        return 0.0 # Return zero similarity when comparison is impossible

    return dot_product / (magnitude_a * magnitude_b) # Return cosine similarity between zero and one


class ScoredVectorStore(VectorStore): # Create an in-memory vector store with scored search

    def __init__(self, embedding: Embeddings) -> None: # Initialize the vector store
        self._embedding = embedding # Store the embedding model
        self.documents: list[Document] = [] # Store all documents
        self.vectors: list[list[float]] = [] # Store vectors corresponding to the documents

    @property # Define the embeddings property
    def embeddings(self) -> Embeddings: # Return the configured embedding model
        return self._embedding # Return the stored embedding model

    @classmethod # Mark the method as a class-level factory
    def from_texts( # Create a vector store from text values
        cls, # Receive the current class
        texts: list[str], # Receive the document texts
        embedding: Embeddings, # Receive the embedding implementation
        metadatas: list[dict[str, Any]] | None = None, # Receive optional document metadata
        *, # Make the following argument keyword-only
        ids: list[str] | None = None, # Receive optional document IDs
        **kwargs: Any, # Receive additional initialization arguments
    ) -> "ScoredVectorStore": # Return the populated vector store
        metadatas = metadatas or [{} for _ in texts] # Create empty metadata when none is supplied
        ids = ids or [f"doc-{index}" for index in range(1, len(texts) + 1)] # Generate IDs when none are supplied
        store = cls(embedding=embedding) # Create an empty vector store
        store.documents = [ # Create Document objects
            Document(id=document_id, page_content=text, metadata=metadata) # Create one Document
            for text, metadata, document_id in zip(texts, metadatas, ids) # Combine matching values
        ] # Finish creating the document list
        store.vectors = embedding.embed_documents(texts) # Generate and store all document vectors
        return store # Return the populated vector store

    def similarity_search_with_score( # Search for documents and return native distance scores
        self, # Receive the current vector-store object
        query: str, # Receive the search query
        k: int = 4, # Receive the maximum number of results
        **kwargs: Any, # Receive additional search arguments
    ) -> list[tuple[Document, float]]: # Return documents paired with distance scores
        query_vector = self._embedding.embed_query(query) # Convert the query into a vector
        scored_documents = [] # Create a list for documents and distances

        for document, document_vector in zip(self.documents, self.vectors): # Process every stored document
            similarity = cosine_similarity(query_vector, document_vector) # Calculate cosine similarity
            distance = 1.0 - similarity # Convert similarity into distance
            scored_documents.append((document, distance)) # Store the document with its distance

        scored_documents.sort(key=lambda item: item[1]) # Sort from lowest distance to highest distance
        return scored_documents[:k] # Return only the top-k closest documents

    def similarity_search( # Search for documents without returning scores
        self, # Receive the current vector-store object
        query: str, # Receive the search query
        k: int = 4, # Receive the maximum number of results
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return the closest documents
        scored_results = self.similarity_search_with_score(query=query, k=k, **kwargs) # Run scored search
        return [document for document, distance in scored_results] # Remove scores and return Documents

    def _select_relevance_score_fn(self) -> Callable[[float], float]: # Provide distance-to-relevance conversion

        def convert_distance_to_relevance(distance: float) -> float: # Convert one distance into relevance
            relevance = 1.0 - distance # Convert lower distance into higher relevance
            return max(0.0, min(1.0, relevance)) # Ensure the result remains between zero and one

        return convert_distance_to_relevance # Return the conversion function

In [ ]:
# Create sample data
vocabulary = [ # Define the embedding dimensions
    "password", # Represent password-related content
    "account", # Represent account-related content
    "refund", # Represent refund-related content
    "payment", # Represent payment-related content
    "delivery", # Represent delivery-related content
    "order", # Represent order-related content
    "login", # Represent login-related content
    "support", # Represent support-related content
] # Finish the vocabulary list

embedding_model = KeywordEmbeddings(vocabulary) # Create the custom embedding model

vector_store = ScoredVectorStore.from_texts( # Create and populate the vector store
    texts=[ # Provide the document text values
        "Reset your account password from the account settings page.", # Add a password document
        "Contact support when your account login fails.", # Add a login-support document
        "Refunds are returned to the original payment method.", # Add a refund document
        "Track your delivery through the order tracking page.", # Add a delivery document
    ], # Finish the text list
    embedding=embedding_model, # Supply the embedding model
    metadatas=[ # Provide metadata for every document
        {"category": "password"}, # Add password metadata
        {"category": "support"}, # Add support metadata
        {"category": "refund"}, # Add refund metadata
        {"category": "delivery"}, # Add delivery metadata
    ], # Finish the metadata list
    ids=["faq-1", "faq-2", "faq-3", "faq-4"], # Provide explicit document IDs
) # Finish creating the vector store

query = "How can I reset my account password?" # Define the search query

In [ ]:
# 1. similarity_search_with_score()
distance_results = vector_store.similarity_search_with_score( # Perform synchronous scored search
    query=query, # Provide the search query
    k=4, # Request four results
) # Finish the search call

print("Native distance scores:") # Display the result heading

for document, distance in distance_results: # Process every scored result
    print(f"\nID: {document.id}") # Display the document ID
    print(f"Distance: {distance:.3f}") # Display the native distance score
    print(f"Content: {document.page_content}") # Display the document content

In [ ]:
# 2. asimilarity_search_with_score()
async_distance_results = await vector_store.asimilarity_search_with_score( # Run scored search asynchronously
    query=query, # Provide the search query
    k=3, # Request three results
) # Finish the asynchronous search

print("\nAsynchronous distance scores:") # Display the result heading

for document, distance in async_distance_results: # Process every asynchronous result
    print(f"{document.id}: {distance:.3f}") # Display the document ID and distance

In [ ]:
# 3. similarity_search_with_relevance_scores()
relevance_results = vector_store.similarity_search_with_relevance_scores( # Perform normalized relevance search
    query=query, # Provide the search query
    k=4, # Request four results
) # Finish the relevance search

print("\nNormalized relevance scores:") # Display the result heading

for document, relevance in relevance_results: # Process every relevance result
    print(f"\nID: {document.id}") # Display the document ID
    print(f"Relevance: {relevance:.3f}") # Display the normalized relevance score
    print(f"Content: {document.page_content}") # Display the document content

In [ ]:
# 4. Apply score_threshold
threshold_results = vector_store.similarity_search_with_relevance_scores( # Perform threshold-based search
    query=query, # Provide the search query
    k=4, # Consider four documents
    score_threshold=0.30, # Keep only documents with sufficient relevance
) # Finish the threshold search

print("\nThreshold results:") # Display the threshold-result heading

for document, relevance in threshold_results: # Process every qualifying result
    print(f"{document.id}: {relevance:.3f} -> {document.page_content}") # Display the score and document

In [ ]:
# 5. asimilarity_search_with_relevance_scores()
async_relevance_results = await vector_store.asimilarity_search_with_relevance_scores( # Run relevance search asynchronously
    query=query, # Provide the search query
    k=3, # Request three results
    score_threshold=0.20, # Apply a minimum relevance score
) # Finish the asynchronous relevance search

print("\nAsynchronous relevance results:") # Display the result heading

for document, relevance in async_relevance_results: # Process every asynchronous relevance result
    print(f"{document.id}: {relevance:.3f}") # Display the document ID and relevance score

In [ ]:
# 6. asimilarity_search()
async_documents = await vector_store.asimilarity_search( # Run normal similarity search asynchronously
    query=query, # Provide the search query
    k=2, # Request two documents
) # Finish the asynchronous search

print("\nAsynchronous documents:") # Display the result heading

for document in async_documents: # Process every retrieved Document
    print(f"{document.id}: {document.page_content}") # Display the document ID and content

## Vector search

### `similarity_search_by_vector`

```python
similarity_search_by_vector(
    self,
    embedding: list[float], # Query embedding vector
    k: int = 4, # Maximum number of documents to return
    **kwargs: Any, # Store-specific search arguments
) -> list[Document] # Documents most similar to the vector
```

The base implementation raises `NotImplementedError`.

### `asimilarity_search_by_vector`

Runs `similarity_search_by_vector` through an executor.

```python
async asimilarity_search_by_vector(
    self,
    embedding: list[float],
    k: int = 4,
    **kwargs: Any,
) -> list[Document]
```

## Maximal marginal relevance search

### `max_marginal_relevance_search`

Selects documents by balancing query similarity and diversity.

```python
max_marginal_relevance_search(
    self,
    query: str, # Query text
    k: int = 4, # Number of selected documents
    fetch_k: int = 20, # Number of candidates considered by MMR
    lambda_mult: float = 0.5, # Diversity control from `0` for maximum diversity to `1` for minimum diversity
    **kwargs: Any, # Store-specific search arguments
) -> list[Document] # Documents selected by maximal marginal relevance
```

The base implementation raises `NotImplementedError`.

### `amax_marginal_relevance_search`

Runs `max_marginal_relevance_search` through an executor.

```python
async amax_marginal_relevance_search(
    self,
    query: str,
    k: int = 4,
    fetch_k: int = 20,
    lambda_mult: float = 0.5,
    **kwargs: Any,
) -> list[Document]
```

### `max_marginal_relevance_search_by_vector`

```python
max_marginal_relevance_search_by_vector(
    self,
    embedding: list[float], # Query embedding vector
    k: int = 4,
    fetch_k: int = 20,
    lambda_mult: float = 0.5,
    **kwargs: Any,
) -> list[Document]
```

The base implementation raises `NotImplementedError`.

### `amax_marginal_relevance_search_by_vector`

Runs `max_marginal_relevance_search_by_vector` through an executor.

```python
async amax_marginal_relevance_search_by_vector(
    self,
    embedding: list[float],
    k: int = 4,
    fetch_k: int = 20,
    lambda_mult: float = 0.5,
    **kwargs: Any,
) -> list[Document]
```

## Factory methods

### `from_documents`

Creates a store by extracting text, metadata, and usable IDs from documents, then delegating to `from_texts`.

```python
@classmethod
from_documents(
    cls,
    documents: list[Document], # Documents used to initialize the store
    embedding: Embeddings, # Embedding implementation to use
    **kwargs: Any, # Store-specific initialization arguments
) -> Self # Initialized vector store
```

### `afrom_documents`

Asynchronous counterpart that delegates to `afrom_texts`.

```python
@classmethod
async afrom_documents(
    cls,
    documents: list[Document],
    embedding: Embeddings,
    **kwargs: Any,
) -> Self
```

### `afrom_texts`

Runs `from_texts` through an executor.

```python
@classmethod
async afrom_texts(
    cls,
    texts: list[str],
    embedding: Embeddings,
    metadatas: list[dict[str, Any]] | None = None,
    *,
    ids: list[str] | None = None,
    **kwargs: Any,
) -> Self
```

## Retriever conversion

### `as_retriever`

Wraps the vector store in a `VectorStoreRetriever`.

```python
as_retriever(
    self,
    **kwargs: Any, # Retriever fields such as `search_type`, `search_kwargs`, and `tags`
) -> VectorStoreRetriever # Retriever backed by this vector store
```

Supported search types are `"similarity"`, `"similarity_score_threshold"`, and `"mmr"`. Search arguments can include `k`, `score_threshold`, `fetch_k`, `lambda_mult`, and store-specific metadata filters.

In [ ]:
#%pip install -U langchain-core # Install LangChain Core if it is unavailable

import math # Import mathematical functions for vector calculations
import re # Import regular expressions for extracting words
from typing import Any # Import Any for additional search arguments
from langchain_core.documents import Document # Import Document for storing text and metadata
from langchain_core.embeddings import Embeddings # Import the abstract Embeddings interface
from langchain_core.vectorstores import VectorStore # Import the abstract VectorStore interface


class KeywordEmbeddings(Embeddings): # Create a predictable keyword-based embedding model

    def __init__(self, vocabulary: list[str]) -> None: # Initialize the embedding model
        self.vocabulary = [word.lower() for word in vocabulary] # Store vocabulary words in lowercase

    def _create_vector(self, text: str) -> list[float]: # Convert one text value into a vector
        words = re.findall(r"\b\w+\b", text.lower()) # Extract lowercase words from the text
        vector = [float(words.count(word)) for word in self.vocabulary] # Count each vocabulary word
        return vector # Return the generated vector

    def embed_documents(self, texts: list[str]) -> list[list[float]]: # Embed multiple documents
        return [self._create_vector(text) for text in texts] # Return one vector for each document

    def embed_query(self, text: str) -> list[float]: # Embed one query
        return self._create_vector(text) # Return the generated query vector


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float: # Calculate cosine similarity
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b)) # Calculate the dot product
    magnitude_a = math.sqrt(sum(value ** 2 for value in vector_a)) # Calculate the first vector magnitude
    magnitude_b = math.sqrt(sum(value ** 2 for value in vector_b)) # Calculate the second vector magnitude

    if magnitude_a == 0 or magnitude_b == 0: # Check whether either vector contains only zeros
        return 0.0 # Return zero similarity when comparison is impossible

    return dot_product / (magnitude_a * magnitude_b) # Return the cosine-similarity value


class MMRVectorStore(VectorStore): # Create an in-memory vector store supporting vector and MMR search

    def __init__(self, embedding: Embeddings) -> None: # Initialize the vector store
        self._embedding = embedding # Store the embedding model
        self.documents: list[Document] = [] # Store all Documents
        self.vectors: list[list[float]] = [] # Store vectors corresponding to the Documents

    @property # Define the embeddings property
    def embeddings(self) -> Embeddings: # Return the configured embedding model
        return self._embedding # Return the stored embedding model

    @classmethod # Define a class-level factory method
    def from_texts( # Create a vector store from text values
        cls, # Receive the current vector-store class
        texts: list[str], # Receive the document texts
        embedding: Embeddings, # Receive the embedding implementation
        metadatas: list[dict[str, Any]] | None = None, # Receive optional metadata
        *, # Make the following parameter keyword-only
        ids: list[str] | None = None, # Receive optional document IDs
        **kwargs: Any, # Receive additional initialization arguments
    ) -> "MMRVectorStore": # Return the initialized vector store
        metadatas = metadatas or [{} for _ in texts] # Create empty metadata when none is supplied
        ids = ids or [f"doc-{index}" for index in range(1, len(texts) + 1)] # Generate IDs when none are supplied
        store = cls(embedding=embedding) # Create an empty vector store
        store.documents = [ # Create Document objects
            Document(id=document_id, page_content=text, metadata=metadata) # Create one Document
            for text, metadata, document_id in zip(texts, metadatas, ids) # Combine matching values
        ] # Finish creating the Document list
        store.vectors = embedding.embed_documents(texts) # Generate vectors for all Documents
        return store # Return the populated vector store

    def similarity_search_by_vector( # Search directly using a supplied vector
        self, # Receive the current vector-store object
        embedding: list[float], # Receive the query embedding vector
        k: int = 4, # Receive the maximum number of results
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return the most similar Documents
        scored_documents = [] # Create a list for Documents and similarity scores

        for document, document_vector in zip(self.documents, self.vectors): # Process every stored Document
            score = cosine_similarity(embedding, document_vector) # Calculate vector similarity
            scored_documents.append((document, score)) # Store the Document with its score

        scored_documents.sort(key=lambda item: item[1], reverse=True) # Sort from highest similarity to lowest
        return [document for document, score in scored_documents[:k]] # Return only the top-k Documents

    def similarity_search( # Search using text instead of a vector
        self, # Receive the current vector-store object
        query: str, # Receive the search query
        k: int = 4, # Receive the maximum number of results
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return the most similar Documents
        query_vector = self._embedding.embed_query(query) # Convert the query into a vector
        return self.similarity_search_by_vector(query_vector, k=k, **kwargs) # Delegate to vector search

    def max_marginal_relevance_search_by_vector( # Perform MMR search using a query vector
        self, # Receive the current vector-store object
        embedding: list[float], # Receive the query embedding vector
        k: int = 4, # Receive the number of final Documents
        fetch_k: int = 20, # Receive the number of initial candidates
        lambda_mult: float = 0.5, # Receive the relevance-versus-diversity balance
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return relevant and diverse Documents
        scored_indices = [] # Create a list for Document indices and query similarities

        for index, document_vector in enumerate(self.vectors): # Process every stored vector
            query_score = cosine_similarity(embedding, document_vector) # Calculate query similarity
            scored_indices.append((index, query_score)) # Store the index with its similarity score

        scored_indices.sort(key=lambda item: item[1], reverse=True) # Sort candidates by query relevance
        candidate_indices = [index for index, score in scored_indices[:fetch_k]] # Keep only fetch_k candidates

        if not candidate_indices: # Check whether no candidates are available
            return [] # Return an empty result list

        selected_indices = [candidate_indices[0]] # Select the most relevant candidate first
        remaining_indices = candidate_indices[1:] # Store the unselected candidate indices

        while remaining_indices and len(selected_indices) < k: # Continue until k Documents are selected
            best_index = None # Create a placeholder for the next selected index
            best_mmr_score = float("-inf") # Begin with the lowest possible MMR score

            for candidate_index in remaining_indices: # Evaluate every remaining candidate
                query_similarity = cosine_similarity( # Calculate candidate relevance to the query
                    embedding, # Supply the query vector
                    self.vectors[candidate_index], # Supply the candidate vector
                ) # Finish calculating query similarity
                selected_similarity = max( # Calculate similarity to the selected Documents
                    cosine_similarity( # Compare the candidate with one selected vector
                        self.vectors[candidate_index], # Supply the candidate vector
                        self.vectors[selected_index], # Supply a selected vector
                    ) # Finish calculating candidate-selected similarity
                    for selected_index in selected_indices # Process all selected Document indices
                ) # Finish finding the highest similarity to selected Documents
                mmr_score = ( # Calculate the final MMR score
                    lambda_mult * query_similarity # Reward relevance to the query
                    - (1.0 - lambda_mult) * selected_similarity # Penalize similarity to selected Documents
                ) # Finish calculating the MMR score

                if mmr_score > best_mmr_score: # Check whether this is the best candidate so far
                    best_mmr_score = mmr_score # Store the new best score
                    best_index = candidate_index # Store the new best candidate index

            selected_indices.append(best_index) # Add the best candidate to the selected indices
            remaining_indices.remove(best_index) # Remove the selected candidate from remaining indices

        return [self.documents[index] for index in selected_indices] # Return the selected Documents

    def max_marginal_relevance_search( # Perform MMR search using text
        self, # Receive the current vector-store object
        query: str, # Receive the search query
        k: int = 4, # Receive the number of final Documents
        fetch_k: int = 20, # Receive the number of initial candidates
        lambda_mult: float = 0.5, # Receive the relevance-versus-diversity balance
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return relevant and diverse Documents
        query_vector = self._embedding.embed_query(query) # Convert the query into a vector
        return self.max_marginal_relevance_search_by_vector( # Delegate to vector-based MMR search
            embedding=query_vector, # Supply the generated query vector
            k=k, # Forward the requested result count
            fetch_k=fetch_k, # Forward the candidate count
            lambda_mult=lambda_mult, # Forward the relevance-diversity balance
            **kwargs, # Forward additional search arguments
        ) # Finish the vector-based MMR search

In [ ]:
# Create the test vector store
vocabulary = [ # Define the embedding dimensions
    "python", # Represent Python-related content
    "programming", # Represent programming-related content
    "langchain", # Represent LangChain-related content
    "llm", # Represent LLM-related content
    "vector", # Represent vector-related content
    "embedding", # Represent embedding-related content
    "database", # Represent database-related content
    "retrieval", # Represent retrieval-related content
] # Finish the vocabulary list

embedding_model = KeywordEmbeddings(vocabulary) # Create the keyword embedding model

vector_store = MMRVectorStore.from_texts( # Create and populate the vector store
    texts=[ # Provide the Document text values
        "Python is a popular programming language.", # Add a general Python Document
        "Python programming is useful for data science.", # Add another similar Python Document
        "LangChain helps developers build LLM applications.", # Add a LangChain Document
        "Vector databases store embedding vectors.", # Add a vector-database Document
        "Retrieval systems find relevant documents.", # Add a retrieval Document
        "Embeddings represent text as numerical vectors.", # Add an embedding Document
    ], # Finish the text list
    embedding=embedding_model, # Supply the embedding model
    metadatas=[ # Provide metadata for each Document
        {"topic": "python"}, # Add metadata for the first Document
        {"topic": "python"}, # Add metadata for the second Document
        {"topic": "langchain"}, # Add metadata for the third Document
        {"topic": "vector-store"}, # Add metadata for the fourth Document
        {"topic": "retrieval"}, # Add metadata for the fifth Document
        {"topic": "embeddings"}, # Add metadata for the sixth Document
    ], # Finish the metadata list
    ids=["doc-1", "doc-2", "doc-3", "doc-4", "doc-5", "doc-6"], # Provide explicit Document IDs
) # Finish creating the vector store

query = "How do vector embeddings help retrieval?" # Define the search query
query_vector = embedding_model.embed_query(query) # Convert the query into a vector

print(f"Query vector: {query_vector}") # Display the generated query vector

In [ ]:
# 1. similarity_search_by_vector()
vector_results = vector_store.similarity_search_by_vector( # Search using the query vector
    embedding=query_vector, # Supply the generated vector
    k=3, # Request the three closest Documents
) # Finish the vector search

print("\nVector search results:") # Display the result heading

for document in vector_results: # Process every vector-search result
    print(f"{document.id}: {document.page_content}") # Display the ID and content

In [ ]:
# 2. asimilarity_search_by_vector()
async_vector_results = await vector_store.asimilarity_search_by_vector( # Run vector search asynchronously
    embedding=query_vector, # Supply the generated query vector
    k=3, # Request three Documents
) # Finish the asynchronous vector search

print("\nAsynchronous vector search results:") # Display the result heading

for document in async_vector_results: # Process every asynchronous result
    print(f"{document.id}: {document.page_content}") # Display the ID and content

In [ ]:
# 3. max_marginal_relevance_search()
mmr_results = vector_store.max_marginal_relevance_search( # Perform text-based MMR search
    query=query, # Supply the search query
    k=3, # Select three final Documents
    fetch_k=6, # Consider six initial candidates
    lambda_mult=0.5, # Balance relevance and diversity equally
) # Finish the MMR search

print("\nText-based MMR results:") # Display the result heading

for document in mmr_results: # Process every MMR-selected Document
    print(f"{document.id}: {document.page_content}") # Display the ID and content

In [ ]:
# 4. max_marginal_relevance_search_by_vector()
vector_mmr_results = vector_store.max_marginal_relevance_search_by_vector( # Perform vector-based MMR search
    embedding=query_vector, # Supply the query vector
    k=3, # Select three final Documents
    fetch_k=6, # Consider six candidate Documents
    lambda_mult=0.3, # Give more importance to diversity
) # Finish the vector-based MMR search

print("\nVector-based MMR results:") # Display the result heading

for document in vector_mmr_results: # Process every selected Document
    print(f"{document.id}: {document.page_content}") # Display the ID and content

In [ ]:
# 5. Asynchronous text-based MMR
async_mmr_results = await vector_store.amax_marginal_relevance_search( # Run text-based MMR asynchronously
    query=query, # Supply the search query
    k=3, # Select three final Documents
    fetch_k=6, # Consider six initial candidates
    lambda_mult=0.5, # Balance relevance and diversity
) # Finish the asynchronous MMR search

print("\nAsynchronous text-based MMR results:") # Display the result heading

for document in async_mmr_results: # Process every asynchronous MMR result
    print(f"{document.id}: {document.page_content}") # Display the ID and content

In [ ]:
# 6. Asynchronous vector-based MMR
async_vector_mmr_results = await vector_store.amax_marginal_relevance_search_by_vector( # Run vector-based MMR asynchronously
    embedding=query_vector, # Supply the query vector
    k=3, # Select three final Documents
    fetch_k=6, # Consider six initial candidates
    lambda_mult=0.3, # Give more importance to diversity
) # Finish the asynchronous vector-based MMR search

print("\nAsynchronous vector-based MMR results:") # Display the result heading

for document in async_vector_mmr_results: # Process every asynchronous result
    print(f"{document.id}: {document.page_content}") # Display the ID and content

# `VectorStoreRetriever: BaseRetriever`

Retriever that delegates document lookup and document addition to a `VectorStore`.

## Fields

```python
vectorstore: VectorStore # Vector store used for retrieval
search_type: str = "similarity" # Search strategy
search_kwargs: dict[str, Any] = Field(default_factory=dict) # Arguments forwarded to vector-store search
allowed_search_types: ClassVar[Collection[str]] = (
    "similarity",
    "similarity_score_threshold",
    "mmr",
) # Supported search strategies
```

## Behaviour

Before model validation, `search_type` must belong to `allowed_search_types`. When it is `"similarity_score_threshold"`, `search_kwargs["score_threshold"]` must exist and be a `float`; otherwise, construction raises `ValueError`.

At retrieval time, call-specific keyword arguments override matching values in `search_kwargs`. The retriever dispatches to synchronous or asynchronous similarity, thresholded relevance, or maximal marginal relevance search on its vector store.

## Methods

### `add_documents`

```python
add_documents(
    self,
    documents: list[Document], # Documents to add to the underlying vector store
    **kwargs: Any, # Store-specific addition arguments
) -> list[str] # IDs assigned by the vector store
```

### `aadd_documents`

```python
async aadd_documents(
    self,
    documents: list[Document],
    **kwargs: Any,
) -> list[str]
```

In [ ]:
#%pip install -U langchain-core # Install LangChain Core if it is unavailable

import re # Import regular expressions for extracting words
from typing import Any # Import Any for additional arguments
from uuid import uuid4 # Import uuid4 for generating document IDs
from langchain_core.documents import Document # Import the LangChain Document class
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.embeddings.fake import DeterministicFakeEmbedding # Import a test embedding model
from langchain_core.vectorstores import VectorStore # Import the abstract VectorStore class
from langchain_core.vectorstores.base import VectorStoreRetriever # Import the vector-store retriever


def extract_words(text: str) -> set[str]: # Convert text into unique lowercase words
    return set(re.findall(r"\b\w+\b", text.lower())) # Extract and return the words


def calculate_similarity(first_text: str, second_text: str) -> float: # Calculate text similarity
    first_words = extract_words(first_text) # Extract words from the first text
    second_words = extract_words(second_text) # Extract words from the second text
    combined_words = first_words.union(second_words) # Find all unique words

    if not combined_words: # Check whether both texts contain no words
        return 0.0 # Return zero similarity

    matching_words = first_words.intersection(second_words) # Find words shared by both texts
    return len(matching_words) / len(combined_words) # Return a normalized score between zero and one


class FAQVectorStore(VectorStore): # Create a simple in-memory FAQ vector store

    def __init__(self, embedding: Embeddings) -> None: # Initialize the vector store
        self._embedding = embedding # Store the embedding model
        self.documents: dict[str, Document] = {} # Store documents using their IDs

    @property # Define the embeddings property
    def embeddings(self) -> Embeddings: # Return the configured embedding model
        return self._embedding # Return the stored embedding model

    def add_documents( # Define synchronous document addition
        self, # Receive the current vector-store object
        documents: list[Document], # Receive documents to add
        **kwargs: Any, # Receive additional store-specific arguments
    ) -> list[str]: # Return the assigned document IDs
        supplied_ids = kwargs.get("ids") # Read optional IDs supplied through keyword arguments
        assigned_ids = [] # Create a list for assigned IDs

        if supplied_ids is not None and len(supplied_ids) != len(documents): # Validate the supplied ID count
            raise ValueError("ID count must match document count.") # Raise an error for mismatched counts

        for index, document in enumerate(documents): # Process every supplied document
            document_id = supplied_ids[index] if supplied_ids else document.id # Prefer IDs supplied through keyword arguments
            document_id = document_id or str(uuid4()) # Generate an ID when one is unavailable

            stored_document = Document( # Create a Document with a guaranteed ID
                id=document_id, # Assign the resolved document ID
                page_content=document.page_content, # Copy the document content
                metadata=document.metadata, # Copy the document metadata
            ) # Finish creating the stored Document

            self.documents[document_id] = stored_document # Store the Document using its ID
            assigned_ids.append(document_id) # Record the assigned ID

        return assigned_ids # Return all assigned IDs

    def similarity_search( # Define normal similarity search
        self, # Receive the current vector-store object
        query: str, # Receive the user query
        k: int = 4, # Receive the maximum result count
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return the most similar Documents
        scored_documents = [ # Create documents paired with similarity scores
            (document, calculate_similarity(query, document.page_content)) # Calculate one document score
            for document in self.documents.values() # Process every stored Document
        ] # Finish creating the scored list

        scored_documents.sort(key=lambda item: item[1], reverse=True) # Sort from highest score to lowest
        return [document for document, score in scored_documents[:k]] # Return only the top-k Documents

    def similarity_search_with_relevance_scores( # Define threshold-compatible relevance search
        self, # Receive the current vector-store object
        query: str, # Receive the user query
        k: int = 4, # Receive the maximum result count
        **kwargs: Any, # Receive optional arguments such as score_threshold
    ) -> list[tuple[Document, float]]: # Return Documents with normalized scores
        score_threshold = kwargs.get("score_threshold") # Read the optional minimum score
        scored_documents = [ # Create documents paired with relevance scores
            (document, calculate_similarity(query, document.page_content)) # Calculate one relevance score
            for document in self.documents.values() # Process every stored Document
        ] # Finish creating the scored list

        scored_documents.sort(key=lambda item: item[1], reverse=True) # Sort from highest relevance to lowest
        scored_documents = scored_documents[:k] # Keep only the top-k results

        if score_threshold is not None: # Check whether threshold filtering was requested
            scored_documents = [ # Create a filtered result list
                (document, score) # Preserve each qualifying Document and score
                for document, score in scored_documents # Process every scored result
                if score >= score_threshold # Keep scores meeting the threshold
            ] # Finish filtering the results

        return scored_documents # Return the normalized relevance results

    def max_marginal_relevance_search( # Define MMR search
        self, # Receive the current vector-store object
        query: str, # Receive the user query
        k: int = 4, # Receive the final number of Documents
        fetch_k: int = 20, # Receive the initial candidate count
        lambda_mult: float = 0.5, # Receive the relevance-diversity balance
        **kwargs: Any, # Receive additional search arguments
    ) -> list[Document]: # Return relevant and diverse Documents
        candidates = self.similarity_search(query=query, k=fetch_k) # Retrieve the most relevant initial candidates

        if not candidates: # Check whether no candidates exist
            return [] # Return an empty list

        selected_documents = [candidates[0]] # Select the most relevant Document first
        remaining_documents = candidates[1:] # Store the remaining candidates

        while remaining_documents and len(selected_documents) < k: # Continue until enough Documents are selected
            best_document = None # Create a placeholder for the next Document
            best_score = float("-inf") # Begin with the lowest possible score

            for candidate in remaining_documents: # Evaluate every remaining candidate
                query_relevance = calculate_similarity(query, candidate.page_content) # Calculate relevance to the query
                redundancy = max( # Calculate similarity to already selected Documents
                    calculate_similarity(candidate.page_content, selected.page_content) # Compare with one selected Document
                    for selected in selected_documents # Process every selected Document
                ) # Finish calculating redundancy
                mmr_score = (lambda_mult * query_relevance) - ((1 - lambda_mult) * redundancy) # Calculate the MMR score

                if mmr_score > best_score: # Check whether this candidate is currently best
                    best_score = mmr_score # Store the new best score
                    best_document = candidate # Store the new best Document

            selected_documents.append(best_document) # Add the selected Document
            remaining_documents.remove(best_document) # Remove it from the remaining candidates

        return selected_documents # Return the MMR-selected Documents

    @classmethod # Define a class-level factory
    def from_texts( # Create the store from text values
        cls, # Receive the vector-store class
        texts: list[str], # Receive the initial texts
        embedding: Embeddings, # Receive the embedding model
        metadatas: list[dict[str, Any]] | None = None, # Receive optional metadata
        *, # Make the following argument keyword-only
        ids: list[str] | None = None, # Receive optional IDs
        **kwargs: Any, # Receive additional initialization arguments
    ) -> "FAQVectorStore": # Return an initialized vector store
        metadatas = metadatas or [{} for _ in texts] # Create empty metadata when none is supplied
        ids = ids or [str(uuid4()) for _ in texts] # Generate IDs when none are supplied

        documents = [ # Create Document objects
            Document(id=document_id, page_content=text, metadata=metadata) # Create one Document
            for text, metadata, document_id in zip(texts, metadatas, ids) # Combine matching values
        ] # Finish creating the Document list

        store = cls(embedding=embedding) # Create an empty vector store
        store.add_documents(documents) # Add the created Documents
        return store # Return the populated vector store


embedding_model = DeterministicFakeEmbedding(size=5) # Create a deterministic test embedding model

vector_store = FAQVectorStore.from_texts( # Create and populate the vector store
    texts=[ # Provide the FAQ text values
        "Reset your account password from the account settings page.", # Add the password-reset FAQ
        "Contact technical support when your account login fails.", # Add the login-support FAQ
        "Refunds are returned to the original payment method.", # Add the refund FAQ
        "Track your delivery through the order tracking page.", # Add the delivery FAQ
    ], # Finish the text list
    embedding=embedding_model, # Supply the embedding model
    metadatas=[ # Provide metadata for each FAQ
        {"category": "password"}, # Add password metadata
        {"category": "support"}, # Add support metadata
        {"category": "refund"}, # Add refund metadata
        {"category": "delivery"}, # Add delivery metadata
    ], # Finish the metadata list
    ids=["faq-1", "faq-2", "faq-3", "faq-4"], # Provide explicit IDs
) # Finish creating the vector store

In [ ]:
# Similarity retriever
similarity_retriever = VectorStoreRetriever( # Create a similarity-search retriever
    vectorstore=vector_store, # Supply the underlying vector store
    search_type="similarity", # Select normal similarity search
    search_kwargs={"k": 2}, # Retrieve two Documents by default
) # Finish creating the retriever

similarity_results = similarity_retriever.invoke( # Run the retriever
    "How do I reset my account password?" # Provide the search query
) # Finish the retrieval call

print("Similarity results:") # Display the result heading

for document in similarity_results: # Process every retrieved Document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Override search_kwargs for one call
single_result = similarity_retriever.invoke( # Run retrieval with a call-specific override
    "My login is not working.", # Provide the search query
    k=1, # Override the default result count
) # Finish the retrieval call

print("\nCall-specific override:") # Display the result heading

for document in single_result: # Process the retrieved Document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Add documents through the retriever
new_documents = [ # Create Documents to add
    Document( # Create a new payment Document
        id="faq-5", # Assign a Document ID
        page_content="Change your payment method from the billing settings.", # Store the FAQ text
        metadata={"category": "payment"}, # Store its metadata
    ) # Finish creating the Document
] # Finish the Document list

added_ids = similarity_retriever.add_documents( # Add Documents through the retriever
    documents=new_documents # Supply the Documents
) # Finish adding the Documents

print(f"\nSynchronously added IDs: {added_ids}") # Display the IDs assigned by the vector store

In [ ]:
# Add documents asynchronously
async_documents = [ # Create Documents for asynchronous addition
    Document( # Create a new account-email Document
        id="faq-6", # Assign a Document ID
        page_content="Change your account email from the profile page.", # Store the FAQ text
        metadata={"category": "profile"}, # Store its metadata
    ) # Finish creating the Document
] # Finish the asynchronous Document list

async_added_ids = await similarity_retriever.aadd_documents( # Add Documents asynchronously
    documents=async_documents # Supply the Documents
) # Finish adding the Documents

print(f"Asynchronously added IDs: {async_added_ids}") # Display the assigned IDs

In [ ]:
# Threshold retriever
threshold_retriever = VectorStoreRetriever( # Create a threshold-based retriever
    vectorstore=vector_store, # Supply the underlying vector store
    search_type="similarity_score_threshold", # Select thresholded relevance search
    search_kwargs={ # Configure threshold-search arguments
        "k": 6, # Consider up to six Documents
        "score_threshold": 0.15, # Keep Documents scoring at least 0.15
    }, # Finish the search arguments
) # Finish creating the threshold retriever

threshold_results = threshold_retriever.invoke( # Run threshold-based retrieval
    "How can I change my account email?" # Provide the search query
) # Finish the retrieval call

print("\nThreshold results:") # Display the result heading

for document in threshold_results: # Process every qualifying Document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# MMR retriever
mmr_retriever = VectorStoreRetriever( # Create an MMR retriever
    vectorstore=vector_store, # Supply the underlying vector store
    search_type="mmr", # Select maximal marginal relevance search
    search_kwargs={ # Configure MMR arguments
        "k": 3, # Select three final Documents
        "fetch_k": 6, # Consider six initial candidates
        "lambda_mult": 0.5, # Balance relevance and diversity
    }, # Finish the MMR arguments
) # Finish creating the MMR retriever

mmr_results = mmr_retriever.invoke( # Run MMR retrieval
    "I need help with my account." # Provide the search query
) # Finish the retrieval call

print("\nMMR results:") # Display the result heading

for document in mmr_results: # Process every MMR-selected Document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Asynchronous retrieval
async_results = await similarity_retriever.ainvoke( # Run the retriever asynchronously
    "Where can I track my delivery?" # Provide the asynchronous query
) # Finish the asynchronous retrieval call

print("\nAsynchronous retrieval results:") # Display the result heading

for document in async_results: # Process every asynchronously retrieved Document
    print(f"{document.id}: {document.page_content}") # Display its ID and content